In [0]:
class OrderRawProcessor:
    def __init__(self):
        self.spark = spark

        self.path = "**"
        self.checkpointlocation_order = "**"

    def read_order_msg(self):
        return (
            self.spark.readStream.table("dev.spark_db.order_msg")
        )

    def schema_fix_order(self , schema_fix_order_msg ):
        return(

            schema_fix_order_msg.selectExpr(
                        "value.unique_no as unique_id",
                        "value.account_number as account_number",
                        "value.asset_external_id as asset_external_id",
                        "value.order_type as order_type",
                        "value.payment_method as payment_method",
                        "value.currency as currency",
                        "value.quantity as quantity",
                        "to_timestamp(value.created_dt) as created_dt"
                        )
            )
        
    def wrtie_to_order(self , schema_fix_order):

        return (
            schema_fix_order.writeStream.format("delta")
                                  .queryName("OrderStreamingQuery")
                                  .outputMode("append")
                                  .option("checkpointLocation" , self.checkpointlocation_order)
                                  .trigger(availableNow = True)
                                  .option("path",self.path)
                                  .toTable("dev.spark_db.Order")
        )

    def main(self):
        order_msg = self.read_order_msg()

        schema_fixed = self.schema_fix_order(order_msg)

        order_query = self.wrtie_to_order(schema_fixed)

        #order_query.awaitTermination

if __name__=='__main__':
    orp = OrderRawProcessor()
    orp.main()



